In [2]:
import time
from time import sleep
import numpy as np
import serial
import matplotlib.pyplot as plt
import seaborn as sns
from IPython import get_ipython
import atexit
import tkinter as tk
from tkinter import filedialog
import threading
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from json import JSONEncoder
import pandas as pd

## Code

In [3]:
class System:
    """Interface for simple serial communication with DCPS and pyrometer"""
    
    def __init__(self, dcps_port, dcps_baud, pyro_port, pyro_baud, valve_port=None, valve_baud=9600):
        """
        - `dcps_port`: serial port where the USB for the DCPS is plugged in. If
                       the docs are followed, this is COM15
        - `dcps_baud`: baud rate for communication with DCPS. Unless it's changed 
                       on the machine, it is 19200
        - `pyro_port`: same but for the pyrometer. If docs are followed, its COM12
        - `pyro_baud`: communication rate with pyrometer. Should be 115200
                       
        """
        self.dcps = serial.Serial(dcps_port, dcps_baud, timeout=1)
        self.pyro = serial.Serial(pyro_port, pyro_baud, timeout=1)
        self.conenct_dcps()

        if valve_port is not None:
            self.valve = serial.Serial(port=valve_port, baudrate=valve_baud, timeout=1) 
            time.sleep(2)  # wait for Arduino to reset
        else: 
            self.valve = None

    def open_valve(self):
        if not self.valve:
            print('open_valve: No valve connected...'); return
        self.valve.write(b'1')    
        return
        
    def close_valve(self):
        if not self.valve:
            print('close_valve: No valve connected...'); return
        self.valve.write(b'0')
        return
        
    def pulse_valve(self, width):
        if not self.valve:
            print('pulse_valve: No valve connected...'); return

        self.valve.write(b'1') 
        time.sleep(width)
        self.valve.write(b'0')
        return

    def write_dcps_no_wait(self, cmd: str):
        """
        Write to DCPS and disregard the response
        """
        self.dcps.reset_input_buffer()
        self.dcps.write(cmd) 

    def write_dcps_and_wait(self, cmd: str):
        """
        Write to DCPS then read until CRLF is returned.
        Intended for use with setting new current so that it can go fast
        """
        self.dcps.reset_input_buffer()
        self.dcps.write(cmd) 
        resp = self.dcps.readline() # read until \n is received, i.e. the output is returned
        return resp.strip()

    def write_dcps_sleep(self,cmd: str, sleep: float):
        """
        Write to DCPS and wait `sleep` seconds before reading response
        Use for testing, this is not best practices for serial comm
        """
        self.dcps.reset_input_buffer()
        self.dcps.write(cmd)
        time.sleep(sleep)
        resp = self.dcps.read_all()
        return resp.strip(b'\r')
        
    def conenct_dcps(self,):
        """
        Send initialization sequence to DCPS
        """
        print(self.write_dcps_and_wait(b'*IDN?\r\n'))

    def mc(self,):   
        """Measure current"""
        return float(self.write_dcps_and_wait(b'MEAS:CURR?\r\n'))
    def mv(self,):   
        """Measure voltage"""
        return float(self.write_dcps_and_wait(b'MEAS:VOLT?\r\n'))

    def fast_pc(self,n: float): 
        """
        PC = program current; 
        Requests new current `n` and doesn't check response from machine
        """
        self.write_dcps_no_wait(f'CURR {n}\r\n'.encode())
    
    def safe_pc(self,n: float):
        """
        PC = program current;
        Set current to `n` and wait until CRLF from machine;
        raises `RuntimeError` if it fails so the failing behavior
        is predictable and able to be controlled 
        """
        response = ''
        try:
            response = self.write_dcps_and_wait(f'CURR {n}\r\n'.encode())
        except Exception as e:
            print(e)
            raise RuntimeError(f'⛔️ requesting {n} amps failed: {response}')    

    def fast_pv(self,n: float): 
        """
        PV = program voltage 
        Requests new voltage `n` and doesn't check response from machine
        """
        self.write_dcps_no_wait(f'VOLT {n}\r\n'.encode())
    
    def safe_pv(self,n: float):
        """
        PV = program voltage 
        Set voltage to `n` and wait until CRLF from machine;
        raises `RuntimeError` if it fails so the failing behavior
        is predictable and able to be controlled 
        """
        try:
            response = self.write_dcps_and_wait(f'VOLT {n}\r\n'.encode())
        except Exception as e:
            print(e)
            raise RuntimeError(f'⛔️ requesting {n} amps failed: {response}')


    def read_temp(self) -> float:
        """Returns floating point temperature from pyrometer"""
        self.pyro.reset_input_buffer()
        self.pyro.write(b'\x01')
        while self.pyro.in_waiting < 2: continue
        
        resp = self.pyro.read_all()
        return ((int.from_bytes(resp) - 1000) / 10)
    
    def close(self):
        """Close the serial ports"""
        self.dcps.close()
        self.pyro.close()
        if self.valve is not None: self.valve.close()        

In [4]:
class Experiment:
    """
    Defines the experimental parameters and allows easy access to measurements at
    precise intervals and swift adjustments
    """
    def __init__(
        self,
        system: System,
        duration,
        dt,
        max_current,
        max_voltage,
        max_temp,
        min_resistance,
        I0,
        startup_power=None,
        startup_probe_current=5.0,
        startup_probe_duration=0.2,
        startup_duration=1.0,
    ):
        self.sys = system  # interface to the physical system
        self.safe_to_run = False  # force user to use the `with Experiment(...) as self` block
        global running

        self.duration = duration
        self.dt = dt
        assert self.dt >= 0.1, f'ERROR: {dt} too low. Must be >= 0.1'
        
        self.iters = int(duration / dt)
        self.i = 0
        self.t = np.arange(0, duration, dt)

        self.MAX_CURRENT = max_current
        assert self.MAX_CURRENT <= 150, f'ERROR: {max_current} too high. Must be <= 150'
            
        self.MAX_VOLTAGE = max_voltage
        assert self.MAX_VOLTAGE <= 25, f'ERROR: {max_voltage} too high. Must be <= 25'

        self.MIN_RESISTANCE = min_resistance
        assert self.MIN_RESISTANCE >= 0.1, f'ERROR: {min_resistance} too low. Must be >= 0.1'

        self.MAX_TEMP = max_temp

        ##### warn the user
        if max_temp > 1500: print(f'WARNING: high max_temp ({max_temp})... ensure this is correct')
        if max_current > 50: print(f'WARNING: high max_current ({max_current})... ensure this is correct')


        self.I0 = I0    # fallback starting current
        self.Ireq = I0  # Ireq = I_requested = requested current

        # Optional resistance-independent startup. The old code held a fixed
        # current for one unlogged second, so samples with different cold
        # resistance received different startup energy.
        self.STARTUP_POWER = startup_power
        self.STARTUP_PROBE_CURRENT = startup_probe_current
        self.STARTUP_PROBE_DURATION = startup_probe_duration
        self.STARTUP_DURATION = startup_duration

        if self.STARTUP_POWER is not None:
            assert self.STARTUP_POWER > 0, 'startup_power must be positive'
            assert self.STARTUP_PROBE_CURRENT > 0, 'startup_probe_current must be positive'
            assert self.STARTUP_PROBE_DURATION >= 0.1, 'startup_probe_duration must be >= 0.1 s'
            assert self.STARTUP_DURATION >= 0, 'startup_duration must be nonnegative'

        self.I_measured  = []
        self.I_requested = []
        self.V_measured  = []
        self.T_measured  = []
        self.R_measured  = []
        self.P_measured = []
        self.time_points = []

        # Filtered channels are used for the live plots and exported
        # beside the raw measurements. Raw data are never overwritten.
        self.T_filtered = []
        self.P_filtered = []
        self.R_filtered = []
        self.DISPLAY_FILTER_TAU_S = 0.40

    def setI(self, Inew): 
        self.Ireq = min(Inew, self.MAX_CURRENT)
        
    def quit(self):
        global running
        running = False
        
        self.sys.write_dcps_and_wait(b'OUTP OFF\r')
        self.sys.safe_pc(0)
        self.safe_to_run = False
        if self.sys.valve is not None: self.sys.close_valve()

        def is_registered(handler):
            callbacks = get_ipython().events.callbacks
            return 'post_run_cell' in callbacks and handler in callbacks['post_run_cell']

        if is_registered(self.quit):
            get_ipython().events.unregister('post_run_cell', self.quit)

    ### Context manager: enforce use within a context manager so that the DCPS 
    #                    is always turned off when a code block finishes or errs
    def __enter__(self):
        self.safe_to_run = True
        return self
    
    def __exit__(self, exc_type=None, exc_value=None, traceback=None):
        self.quit()
        # always reset the power supply when a resource block finishes
    
    def run(self, controller, verbose=True):
        """
        yields (iteration number, I, V, R, Temp, elapsed time, ∆t since last interval)
        at precise intervals of length `self.dt` until the experiment has elapsed the 
        length of `self.duration` or is finished early.

        After the body of the for loop finishes, `Ireq` current will be requested. Use 
        `setI(...)` to request a specific amount of current.

        User can access other state variables through the object
        """
        assert self.safe_to_run, 'Must use within a `with Experiment(...) as ___:` block' 
        global running

        get_ipython().events.register('post_run_cell', self.quit) 
        atexit.register(quit)  # calls quit for clean completion 

        sys = self.sys

        print('Turning on DCPS Output...', end='  ')
        sys.write_dcps_and_wait(b'OUTP ON\r')
        print('✅')
        print('Initializing Current... ', end='  ')
        sys.safe_pv(self.MAX_VOLTAGE)  # always set this first so voltage is not limiting

        if self.STARTUP_POWER is None:
            # Backward-compatible fixed-current startup.
            sys.safe_pc(min(self.Ireq, self.MAX_CURRENT))
            sleep(1)
        else:
            # Brief low-current resistance probe, followed by a fixed-power
            # startup interval. This makes startup energy approximately
            # independent of the sample's cold resistance.
            probe_current = min(
                self.STARTUP_PROBE_CURRENT,
                self.MAX_CURRENT,
            )
            sys.safe_pc(probe_current)
            sleep(self.STARTUP_PROBE_DURATION)

            Iprobe = sys.mc()
            Vprobe = sys.mv()
            if Iprobe <= 0:
                raise RuntimeError('Startup resistance probe returned zero current')

            Rprobe = Vprobe / Iprobe
            if Rprobe < self.MIN_RESISTANCE:
                raise RuntimeError(
                    f'Startup resistance too low: {Rprobe:.4f} Ohm'
                )

            startup_current = np.sqrt(self.STARTUP_POWER / Rprobe)
            startup_current = min(startup_current, self.MAX_CURRENT)
            sys.safe_pc(startup_current)
            self.Ireq = startup_current
            sleep(self.STARTUP_DURATION)

            print(
                f'R0 = {Rprobe:.4f} Ohm, '
                f'Pstartup = {self.STARTUP_POWER:.1f} W, ',
                end='',
            )

        print('I =', sys.mc(), 'V =', sys.mv())

        axs, Tline, Pline, Rline = controller.axs, controller.Tline, controller.Pline, controller.Rline
        def update(time_width=20):    
            x = np.array(self.time_points)
            T = np.array(self.T_filtered)
            P = np.array(self.P_filtered)
            R = np.array(self.R_filtered)

            if len(x) <= 1: return 
            
            Tline.set_data(x, T)
            Pline.set_data(x, P)
            Rline.set_data(x, R)
            
            for ax, y in zip(axs, [T, P, R]):
                minx = max(x[-1] - time_width, 0)
                maxx = x[-1]
                x_visible = np.argwhere(x >= minx)
                ax.set_ylim(min(y[x_visible])*0.95, max(y[x_visible])*1.05)
                ax.set_xlim(minx, maxx)

            controller.canvas.draw_idle()

        self.start = time.perf_counter()
        self.next_time = time.perf_counter()
        while running:
            Imeas, Vmeas = sys.mc(), sys.mv()
            Rmeas = Vmeas / Imeas
            Pmeas = Imeas * Vmeas

            self.I_measured.append(Imeas)
            self.I_requested.append(self.Ireq)
            self.V_measured.append(Vmeas)
            self.R_measured.append(Rmeas)
            self.P_measured.append(Pmeas)

            Tmeas = sys.read_temp()
            self.T_measured.append(Tmeas)

            # Time-based EMA. Using a time constant keeps the filtering
            # behavior consistent when dt is changed.
            display_alpha = 1.0 - np.exp(
                -self.dt / self.DISPLAY_FILTER_TAU_S
            )
            if len(self.T_filtered) == 0:
                Tdisplay = Tmeas
                Pdisplay = Pmeas
                Rdisplay = Rmeas
            else:
                Tdisplay = (
                    self.T_filtered[-1]
                    + display_alpha * (Tmeas - self.T_filtered[-1])
                )
                Pdisplay = (
                    self.P_filtered[-1]
                    + display_alpha * (Pmeas - self.P_filtered[-1])
                )
                Rdisplay = (
                    self.R_filtered[-1]
                    + display_alpha * (Rmeas - self.R_filtered[-1])
                )

            self.T_filtered.append(Tdisplay)
            self.P_filtered.append(Pdisplay)
            self.R_filtered.append(Rdisplay)

            t_elapsed = time.perf_counter() - self.start
            self.time_points.append(t_elapsed)  
            if t_elapsed > self.duration: break
                # track the true time to that these measurements were made

            #### Update GUI ####            
            controller.root.after(0, update)  # update live plots
            controller.root.after(0, lambda: controller.temp_lab.config(text=f'Temperature: {Tdisplay:.2f}'))
            controller.root.after(0, lambda: controller.power_lab.config(text=f'Power: {Pdisplay:.2f}'))
            controller.root.after(0, lambda: controller.res_lab.config(text=f'Resistance: {Rdisplay:.4f}'))

            ### SAFETY CHECKS ### 
            def safety_check(condition, message): 
                if condition:
                    print(message)
                    self.quit()

            safety_check(
                Tmeas >= self.MAX_TEMP, f'High temp --> cutting supply: {Tmeas} °C > {self.MAX_TEMP} °C'
            )
            safety_check(
                Rmeas < self.MIN_RESISTANCE, f'Low resistance --> cutting supply: {Rmeas} Ohm < {self.MIN_RESISTANCE} Ohm'
            )
            safety_check(
                Vmeas > self.MAX_VOLTAGE, f'High voltage --> cutting supply: {Vmeas} V > {self.MAX_VOLTAGE} V'
            )
            safety_check(
                Imeas > self.MAX_CURRENT, f'High current --> cutting supply: {Imeas} A > {self.MAX_CURRENT} A'
            )

            #############################################################
            ####### User reads the state and modifies it as needed ######
            
            dt = t_elapsed - self.time_points[-2] if self.i > 1 else t_elapsed
            if verbose and self.i > 1:
                print(f'(i={self.i}) t = {t_elapsed:.2f} | ∆t = {dt:.2f} --> T = {Tmeas:.2f}\tI_req = {self.Ireq:.2f}\tI = {Imeas:.2f}\tV = {Vmeas:.2f}\tR = {Rmeas:.2f}\tP = {Imeas*Vmeas:.2f}')
            yield self.i, Imeas, Vmeas, Rmeas, Tmeas, t_elapsed, dt
            sys.fast_pc(min(self.Ireq, self.MAX_CURRENT)) #! changed from safe_pc to fast_pc

            #############################################################

            ####### Sleep until next time step ########
            self.next_time += self.dt
            time_left = self.next_time - time.perf_counter()
            if time_left > 0:
                sleep(time_left)
            else:
                self.next_time = time.perf_counter()
                print(f'⚠️ Warning: {time_left} seconds behind')
            
            self.i += 1

        running = False
        self.quit()
    
    def to_dict(self):
        return {
            "duration": self.duration,
            "MAX_CURRENT": self.MAX_CURRENT,
            "MAX_TEMP": self.MAX_TEMP,
            "MAX_VOLTAGE": self.MAX_VOLTAGE,
            "MIN_RESISTANCE": self.MIN_RESISTANCE,
            "I0": self.I0,
            "STARTUP_POWER": self.STARTUP_POWER,
            "STARTUP_PROBE_CURRENT": self.STARTUP_PROBE_CURRENT,
            "STARTUP_PROBE_DURATION": self.STARTUP_PROBE_DURATION,
            "STARTUP_DURATION": self.STARTUP_DURATION,
            "dt": self.dt,
            "time_points": self.time_points,
            "T_measured": self.T_measured,
            "P_measured": self.P_measured,
            "R_measured": self.R_measured,
            "V_measured": self.V_measured,
            "I_measured": self.I_measured,
            "I_requested": self.I_requested,
            "T_filtered": self.T_filtered,
            "P_filtered": self.P_filtered,
            "R_filtered": self.R_filtered,
        }


In [5]:
class ExperimentWrapper:
    def run(self, controller, verbose=True):
        pass

    def to_dict(self):
        return

In [6]:
class MultiPhaseExperiment(ExperimentWrapper):
    def __init__(
            self, exp: Experiment, 
            eq_temperatures, eq_durations, 
            heating_powers, ma_len, temp_error,
            preemptive_coef, ft_above, ft_below,
            ramping_rate
        ):
        global running
        running = False
        self.exp = exp
        self.eq_temps = eq_temperatures
        self.eq_durations  = eq_durations
        self.heating_powers = heating_powers
        assert len(eq_temperatures) == len(eq_durations) and len(eq_durations) == len(heating_powers)
        self.Pgoal = heating_powers[0]

        self.ma_len = ma_len
        self.TEMP_ERROR = temp_error
        self.PREEMPTIVE_COEF = preemptive_coef
        self.FINETUNE_ABOVE = ft_above
        self.FINETUNE_BELOW = ft_below
        self.power_ramping_rate = ramping_rate

    def run(self, controller, verbose=True):
        global running
        with self.exp as exp:
            phase = 0
            Tgoal = self.eq_temps[phase]
            prev_Tgoal = 0

            controller.root.after(
                0, lambda: controller.eq_lab.config(text=f'Temperature Not Stable (goal = {Tgoal} °C)', bg='tomato')
            )
 
            self.Pgoal = self.heating_powers[phase]
            tmp_Pgoal = None
            is_ramping_power = False

            Tprev = None  
                #! settting to the minimum temp on the pyrometer is a very
                #! bad idea because it doesn't scale to new machines and it
                #! will produce erroneously high heating rates for high current
            ma = []; is_at_eq = False; eq_time = None
            for i, I, V, R, T, elapsed, dt in exp.run(controller, verbose): 
                # 0. track moving average temperature
                ma.append(T)
                if len(ma) > self.ma_len: ma.pop(0)

                if abs(np.mean(ma) - Tgoal) < self.TEMP_ERROR and not is_at_eq: 
                    # Check for equilbrium and switch modes
                    is_at_eq = True
                    eq_time = time.perf_counter()
                    controller.root.after(
                        0, lambda: controller.eq_lab.config(bg="lightgreen", text=f'Temperature Stable (goal = {Tgoal} °C)')
                    )
                    print(f'HIT EQUILIBRIUM: goal temperature = {Tgoal}')

                # 0.5 Ramp up to high powers to not shock the system
                if tmp_Pgoal is not None and is_ramping_power: 
                    tmp_Pgoal += self.power_ramping_rate
                    self.Pgoal = tmp_Pgoal
                
                if self.Pgoal > 200 and tmp_Pgoal is None and not is_ramping_power:
                    # start at 200 watts then ramp up to the goal
                    is_ramping_power = True
                    tmp_Pgoal = 200
                    self.Pgoal = tmp_Pgoal

                if not is_at_eq:
                # 1. preemptive power reduction before equilibrium
                    if Tprev is None: Tprev = T
                        # super important: for very fast heating it is not safe to set 
                        # Tprev to 200 (minumum pyrometer temp) because the first iteration
                        # may be 315 C which will make the rate too high and kill the power
                    Trate = (T - Tprev)/dt
                    Tpred = T + Trate    # really should be Trate*dt to give units of [T] instead of [T/s]
                    pred_error = abs(Tpred - Tgoal)
                    if   Tgoal > prev_Tgoal and Tpred > Tgoal:
                        self.Pgoal -= max(0, self.PREEMPTIVE_COEF * pred_error)
                    elif Tgoal < prev_Tgoal and Tpred < Tgoal:
                        print('WARNING: controlled decreases in temperature not supported')
                        self.Pgoal += self.PREEMPTIVE_COEF * pred_error

                    Tprev = T
                else:
                # 2. fine tune power to sustain perfect temperature
                    error = abs(Tgoal - T)
                    if   T < Tgoal: self.Pgoal += self.FINETUNE_BELOW * error
                    elif T > Tgoal: self.Pgoal -= self.FINETUNE_ABOVE * error


                # Switch phases if done with a phase
                now = time.perf_counter()
                if eq_time != None and now - eq_time > self.eq_durations[phase]: 
                    ma = []
                    is_at_eq = False
                    eq_time = None
                    phase += 1
                    if phase == len(self.eq_temps):
                        running = False
                        print('Successful completion')
                        return

                    prev_Tgoal = self.eq_temps[phase-1]
                    Tgoal = self.eq_temps[phase]
                    controller.root.after(
                        0, lambda: controller.eq_lab.config(text=f'Temperature Not Stable (goal = {Tgoal} °C)', bg='tomato')
                    )
                    
                    self.Pgoal = self.heating_powers[phase]

                exp.setI(np.sqrt(self.Pgoal / R))

            controller.root.after(
                0, lambda: controller.eq_lab.config(bg="lightgray", text=f'Temperature Stability')
            )
    
    def to_dict(self):
        return {
            'NAME': 'MultiPhaseExperiment',
            "eq_temps": self.eq_temps,
            "eq_durations ": self.eq_durations ,
            "heating_powers": self.heating_powers,
            "ma_len": self.ma_len,
            "TEMP_ERROR": self.TEMP_ERROR,
            "PREEMPTIVE_COEF": self.PREEMPTIVE_COEF,
            "FINETUNE_ABOVE": self.FINETUNE_ABOVE,
            "FINETUNE_BELOW": self.FINETUNE_BELOW,
            **self.exp.to_dict(),
        }
    
    def data_to_csv(self, path):
        return pd.DataFrame.from_dict({
            "time_points": self.exp.time_points,
            "T_measured": self.exp.T_measured,
            "P_measured": self.exp.P_measured,
            "R_measured": self.exp.R_measured,
            "V_measured": self.exp.V_measured,
            "I_measured": self.exp.I_measured,
            "I_requested": self.exp.I_requested,
        }, orient='columns').to_csv(path)
    
    def params_to_csv(self, path):
        return pd.DataFrame.from_dict({
            'NAME': 'MultiPhaseExperiment',
            "eq_temps": self.eq_temps,
            "eq_durations ": self.eq_durations ,
            "heating_powers": self.heating_powers,
            "ma_len": self.ma_len,
            "TEMP_ERROR": self.TEMP_ERROR,
            "PREEMPTIVE_COEF": self.PREEMPTIVE_COEF,
            "FINETUNE_ABOVE": self.FINETUNE_ABOVE,
            "FINETUNE_BELOW": self.FINETUNE_BELOW,
            "duration": self.exp.duration,
            "MAX_CURRENT": self.exp.MAX_CURRENT,
            "MAX_TEMP": self.exp.MAX_TEMP,
            "MAX_VOLTAGE": self.exp.MAX_VOLTAGE,
            "MIN_RESISTANCE": self.exp.MIN_RESISTANCE,
            "I0": self.exp.I0,
            "dt": self.exp.dt,
        }, orient='columns').to_csv(path)

In [7]:
class SinglePhaseExperiment(ExperimentWrapper):
    def __init__(
            self, exp: Experiment, 
            eq_temperature, eq_duration, 
            heating_power, ma_len, temp_error,
            preemptive_coef, ft_above, ft_below, 
            ramping_rate
        ):
        self.eq_temp = eq_temperature
        self.eq_duration = eq_duration
        self.heating_power = heating_power
        self.ma_len = ma_len
        self.TEMP_ERROR = temp_error
        self.PREEMPTIVE_COEF = preemptive_coef
        self.FINETUNE_ABOVE = ft_above
        self.FINETUNE_BELOW = ft_below
        
        self.exp = MultiPhaseExperiment(
            exp, [eq_temperature], [eq_duration], [heating_power], 
            ma_len, temp_error, preemptive_coef, ft_above, ft_below,
            ramping_rate
        )

    def run(self, verbose=True):
        self.exp.run(verbose)
    
    def to_dict(self):
        return {
            'NAME': 'SinglePhaseExperiment',
            "eq_temp": self.eq_temp,
            "eq_duration ": self.eq_duration ,
            "heating_power": self.heating_power,
            "ma_len": self.ma_len,
            "TEMP_ERROR": self.TEMP_ERROR,
            "PREEMPTIVE_COEF": self.PREEMPTIVE_COEF,
            "FINETUNE_ABOVE": self.FINETUNE_ABOVE,
            "FINETUNE_BELOW": self.FINETUNE_BELOW,
            **self.exp.to_dict(),
        }

In [8]:
class ExperimentController:
    def __init__(self, wrapper: ExperimentWrapper):
        self.wrapper = wrapper
        self.exp: Experiment = wrapper.exp
        self.sys: System = self.exp.sys
        fig, axs, Tline, Pline, Rline = self.open_graphs()
        self.fig = fig
        self.axs = axs
        self.Tline = Tline
        self.Pline = Pline
        self.Rline = Rline

    def open_graphs(self):
        plt.close()
        fig, axs = plt.subplots(nrows=3, sharex=True)
        ax = axs[0]
        Tline, = ax.plot([], [],)
        ax.set_ylabel('Temperature (°C)')
        ax = axs[1]
        Pline, = ax.plot([], [],color='green')
        ax.set_ylabel('Power (Watts)')
        ax = axs[2]
        Rline, = ax.plot([], [],color='orange')
        ax.set_ylabel(r'Resistance $(\Omega)$')
        ax.set_xlabel('Time (s)')
        fig.tight_layout()
        fig.set_figwidth(11)
        fig.set_figheight(5)
        fig.subplots_adjust(hspace=0.05)
        return fig, axs, Tline, Pline, Rline
    
    def run(self):
        global running
        running = False
        try:
            thread = None
            def start():
                global running
                if running: return
                running = True
                print("Initializing Experiment...")

                # download_data_btn.config(state='disabled')
                # download_params_btn.config(state='disabled')
                # download_exp_btn.config(state='disabled')
                # save_fig_btn.config(state='disabled')
                thread = threading.Thread(target=lambda: self.wrapper.run(self))
                thread.start()
                
            def stop():
                global running
                if running: print("Stopping...")
                running = False
                if thread is not None: thread.join()

                root.destroy()
                def is_registered(handler):
                    callbacks = get_ipython().events.callbacks
                    return 'post_run_cell' in callbacks and handler in callbacks['post_run_cell']
                if is_registered(self.exp.quit): 
                    get_ipython().events.unregister('post_run_cell', self.exp.quit)

            self.root = tk.Tk()
            root = self.root
            root.title("Automated Joule Heating")
            root.geometry("1500x800+10+10")

            ###### Box for start and stop buttons #####
            control_btns = tk.Frame(root)
            control_btns.pack(pady=5)

            start_button = tk.Button(
                control_btns, font=(None, 13), text="Start", command=start, width=20, bg="lightgreen"
            )
            start_button.pack(padx=5, side=tk.LEFT)

            stop_button = tk.Button(
                control_btns, font=(None, 13), text="Stop & Quit", command=stop, width=20, bg="tomato"
            )
            stop_button.pack(padx=5, side=tk.RIGHT)


            ##### Real time values of temp, power, res #####
            live_values_frame = tk.Frame(root)
            live_values_frame.pack(pady=5)

            self.eq_lab = tk.Label(live_values_frame, text='Temperature Stability', bg='lightgray')
            self.eq_lab.pack(padx=5, side=tk.LEFT)
            self.temp_lab = tk.Label(live_values_frame, text='Temperature: ', font=(None, 15))
            self.temp_lab.pack(padx=5, side=tk.LEFT)
            self.power_lab = tk.Label(live_values_frame, text='Power: ', font=(None, 15))
            self.power_lab.pack(padx=5, side=tk.LEFT)
            self.res_lab = tk.Label(live_values_frame, text='Resistance: ', font=(None, 15))
            self.res_lab.pack(padx=5, side=tk.LEFT)

            ##### Real time plot #####
            self.canvas = FigureCanvasTkAgg(self.fig, master=root)
            self.canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)
            self.canvas.draw()

            downloads = tk.Frame(root)
            downloads.pack(pady=5)

            def download_data():
                global running
                if running: return

                path = filedialog.asksaveasfilename(
                    defaultextension=".csv",
                    filetypes=[("CSV File", "*.csv")],
                    title="Download Experiment Data"
                )

                if path:
                    self.wrapper.data_to_csv(path)
                    print(f"Data saved to {path}")

            download_data_btn = tk.Button(downloads, text="Download Data", command=download_data, width=20)
            download_data_btn.pack(padx=5, side=tk.LEFT)

            def download_params():
                global running
                if running: return

                path = filedialog.asksaveasfilename(
                    defaultextension=".csv",
                    filetypes=[("CSV File", "*.csv")],
                    title="Download Experiment Data"
                )

                if path:
                    self.wrapper.data_to_csv(path)
                    print(f"Data saved to {path}")

            download_params_btn = tk.Button(downloads, text="Download Parameters", command=download_params, width=20)
            download_params_btn.pack(padx=5, side=tk.LEFT)

            def download_exp():
                global running
                if running: return

                data = self.wrapper.to_dict()
                data = JSONEncoder().encode(data)

                path = filedialog.asksaveasfilename(
                    defaultextension=".json",
                    filetypes=[("JSON File", "*.json")],
                    title="Download Experiment"
                )

                if path:
                    with open(path, "w") as file:
                        file.write(data)
                    print(f"Experiment saved to {path}")

            download_exp_btn = tk.Button(downloads, text="Download Exp + Params", command=download_exp, width=20)
            download_exp_btn.pack(padx=5, side=tk.LEFT)


            #### Figure manipulation ####

            fig_frame = tk.Frame(root)
            fig_frame.pack(pady=5)

            def save_fig():
                global running
                if running: return

                path = filedialog.asksaveasfilename(
                    defaultextension=".png",
                    filetypes=[("PNG File", "*.png")],
                    title="Download Figure"
                )

                if path:
                    self.fig.savefig(path)
                    print(f"Data saved to {path}")

            save_fig_btn = tk.Button(fig_frame, text="Save Figure", command=save_fig, width=20)
            save_fig_btn.pack(padx=5, side=tk.RIGHT)

            def recenter():
                global running
                if running: return
                for ax, y in zip(self.axs, [self.exp.T_measured, self.exp.P_measured, self.exp.R_measured]):
                    ax.set_xlim(0, max(self.exp.time_points))
                    ax.set_ylim(min(y)*0.95, max(y)*1.05)
                self.canvas.draw()

            recenter_btn = tk.Button(fig_frame, text="Recenter Axes", command=recenter, width=20)
            recenter_btn.pack(padx=5, side=tk.RIGHT)

            def clear():
                for ax in self.axs:
                    for line in ax.lines:
                        line.remove()
                self.canvas.draw()

            clear_btn = tk.Button(fig_frame, text='Clear Figure', width=20, command=clear)
            clear_btn.pack(padx=5, side=tk.RIGHT)

            ##### Valve control ######
            pulse_frame = tk.Frame(root)
            pulse_frame.pack(pady=10)

            lab = tk.Label(pulse_frame, text='Induce Pressure', font=(None, 15))
            lab.pack(pady=10)

            def pulse():
                try:
                    w = float(pulse_width.get())
                    print(f'Pulse width: {w}')
                    threading.Thread(target=lambda: self.sys.pulse_valve(w)).start()
                except: return

            pulse_button = tk.Button(pulse_frame, text="Pulse Duration --> ", command=pulse)
            pulse_button.pack(side=tk.LEFT)

            pulse_width = tk.Entry(pulse_frame, width=10)
            pulse_width.pack(side=tk.RIGHT, padx=5)

            valve_frame = tk.Frame(root)
            valve_frame.pack(pady=5)

            valve_open = tk.Button(valve_frame, text="Open Valve", command=self.sys.open_valve)
            valve_open.pack(padx=10, side=tk.LEFT)

            valve_close = tk.Button(valve_frame, text="Close Valve", command=self.sys.close_valve)
            valve_close.pack(padx=10, side=tk.RIGHT)

            root.protocol("WM_DELETE_WINDOW", stop)
            root.mainloop()
        except Exception as e:
            print(f'ERROR: {e}')
            print('Quitting...')
            stop()

In [9]:
class Power:
    """
    Parent object for defining how the power should change
    between stages in an experiment with multiple phases.
    """
    def __init__(self, v: float):
        self.value = v


class DeltaPower(Power):
    def __init__(self, v: float):
        self.value = v

    def __str__(self):
        if self.value == 0:
            return '0'
        return f"+{self.value}" if self.value > 0 else f"-{self.value}"


class ConstantPower(Power):
    def __init__(self, v: float):
        self.value = v

    def __str__(self):
        return str(self.value)


class Stage:
    def __init__(self, power: Power, temp: float, duration: float):
        self.power = power
        self.temp = temp
        self.duration = duration


class TimedProfileStage(Stage):
    """
    First-stage controller with three explicit modes:

    PREHEAT:
        Used while the pyrometer is at its 200 °C floor.

    TRACK:
        Tracks the reference-shaped temperature trajectory and aims to
        reach 320 °C at target_time.

    HOLD:
        Starts once target_time has passed and filtered temperature reaches
        hold_entry_temp. It holds near 320 °C for exactly duration seconds.
    """
    def __init__(
        self,
        power: Power,
        temp: float,
        duration: float,
        profile_times,
        profile_temps,
        desired_valid_time: float,
        target_time: float,
        preheat_power: float,
        min_power: float,
        max_power: float,
        hold_power: float,
        kp_temp: float = 0.9,
        kp_hold: float = 1.0,
        brake_temp: float = 314.0,
        overshoot_power: float = 65.0,
        max_stage_temp: float = 330.0,
        slew_up_w_per_s: float = 6.0,
        slew_down_w_per_s: float = 30.0,
        late_boost_w_per_s: float = 4.0,
        valid_temp: float = 200.5,
        resistance_filter_alpha: float = 0.08,
        current_slew_up_a_per_s: float = 3.0,
        current_slew_down_a_per_s: float = 15.0,
        temp_filter_alpha: float = 0.15,
        temp_deadband: float = 0.5,
        hold_deadband: float = 0.35,
        hold_entry_temp: float = 319.5,
        control_update_s: float = 0.20,
        linear_target: bool = False,
        kp_endpoint: float = 0.20,
        endpoint_correction_limit_w: float = 12.0,
        endpoint_error_limit_c: float = 50.0,
        rate_filter_tau_s: float = 2.0,
        endpoint_ramp_in_s: float = 3.0,
    ):
        super().__init__(power, temp, duration)

        profile_times = np.asarray(profile_times, dtype=float)
        profile_temps = np.asarray(profile_temps, dtype=float)

        if len(profile_times) != len(profile_temps):
            raise ValueError(
                "profile_times and profile_temps must have the same length"
            )
        if len(profile_times) < 2:
            raise ValueError("At least two trajectory points are required")
        if np.any(np.diff(profile_times) <= 0):
            raise ValueError("profile_times must be strictly increasing")
        if abs(profile_temps[-1] - temp) > 1e-9:
            raise ValueError(
                "The final profile temperature must equal the stage temperature"
            )

        self.profile_fraction_time = (
            (profile_times - profile_times[0])
            / (profile_times[-1] - profile_times[0])
        )
        self.profile_fraction_temp = (
            (profile_temps - profile_temps[0])
            / (profile_temps[-1] - profile_temps[0])
        )

        self.desired_valid_time = float(desired_valid_time)
        self.target_time = float(target_time)
        self.preheat_power = float(preheat_power)
        self.min_power = float(min_power)
        self.max_power = float(max_power)
        self.hold_power = float(hold_power)

        self.kp_temp = float(kp_temp)
        self.kp_hold = float(kp_hold)
        self.brake_temp = float(brake_temp)
        self.overshoot_power = float(overshoot_power)
        self.max_stage_temp = float(max_stage_temp)

        self.slew_up_w_per_s = float(slew_up_w_per_s)
        self.slew_down_w_per_s = float(slew_down_w_per_s)
        self.late_boost_w_per_s = float(late_boost_w_per_s)
        self.valid_temp = float(valid_temp)

        self.resistance_filter_alpha = float(resistance_filter_alpha)
        self.current_slew_up_a_per_s = float(
            current_slew_up_a_per_s
        )
        self.current_slew_down_a_per_s = float(
            current_slew_down_a_per_s
        )
        self.temp_filter_alpha = float(temp_filter_alpha)
        self.temp_deadband = float(temp_deadband)
        self.hold_deadband = float(hold_deadband)
        self.hold_entry_temp = float(hold_entry_temp)
        self.control_update_s = float(control_update_s)

        # First-stage-only straight-line and endpoint-prediction settings.
        self.linear_target = bool(linear_target)
        self.kp_endpoint = float(kp_endpoint)
        self.endpoint_correction_limit_w = float(
            endpoint_correction_limit_w
        )
        self.endpoint_error_limit_c = float(
            endpoint_error_limit_c
        )
        self.rate_filter_tau_s = float(rate_filter_tau_s)
        self.endpoint_ramp_in_s = float(endpoint_ramp_in_s)

        if not (self.min_power <= self.preheat_power <= self.max_power):
            raise ValueError(
                "preheat_power must be between min_power and max_power"
            )
        if not (self.min_power <= self.hold_power <= self.max_power):
            raise ValueError(
                "hold_power must be between min_power and max_power"
            )
        if not (0 < self.resistance_filter_alpha <= 1):
            raise ValueError(
                "resistance_filter_alpha must be in (0, 1]"
            )
        if not (0 < self.temp_filter_alpha <= 1):
            raise ValueError("temp_filter_alpha must be in (0, 1]")
        if self.target_time <= self.desired_valid_time:
            raise ValueError(
                "target_time must be greater than desired_valid_time"
            )
        if not (self.brake_temp < self.temp):
            raise ValueError("brake_temp must be below target temperature")
        if self.hold_entry_temp > self.temp:
            raise ValueError(
                "hold_entry_temp must not exceed target temperature"
            )
        if self.kp_endpoint < 0:
            raise ValueError("kp_endpoint must be non-negative")
        if self.endpoint_correction_limit_w < 0:
            raise ValueError(
                "endpoint_correction_limit_w must be non-negative"
            )
        if self.endpoint_error_limit_c <= 0:
            raise ValueError(
                "endpoint_error_limit_c must be positive"
            )
        if self.rate_filter_tau_s <= 0:
            raise ValueError("rate_filter_tau_s must be positive")
        if self.endpoint_ramp_in_s <= 0:
            raise ValueError("endpoint_ramp_in_s must be positive")

    def target_temperature(self, elapsed, anchor_time, anchor_temp):
        # For linear control, start the trajectory from the ACTUAL first
        # valid pyrometer time. A sample first seen at 5 s therefore gets
        # a gentler slope than one first seen at 10 s, but both lines end
        # at the same target_time.
        trajectory_start_time = anchor_time

        span = self.target_time - trajectory_start_time
        if span <= 0:
            return self.temp

        u = np.clip(
            (elapsed - trajectory_start_time) / span,
            0.0,
            1.0,
        )

        if self.linear_target:
            temp_fraction = u
        else:
            temp_fraction = np.interp(
                u,
                self.profile_fraction_time,
                self.profile_fraction_temp,
            )

        return float(
            anchor_temp + temp_fraction * (self.temp - anchor_temp)
        )

    def clip_power(self, power):
        return float(np.clip(power, self.min_power, self.max_power))

    def slew_limited_power(self, current_power, requested_power, dt):
        requested_power = self.clip_power(requested_power)

        if requested_power > current_power:
            return min(
                requested_power,
                current_power + self.slew_up_w_per_s * dt,
            )

        return max(
            requested_power,
            current_power - self.slew_down_w_per_s * dt,
        )

    def slew_limited_current(self, current, requested, dt):
        if requested > current:
            return min(
                requested,
                current + self.current_slew_up_a_per_s * dt,
            )

        return max(
            requested,
            current - self.current_slew_down_a_per_s * dt,
        )


class SequentialExperiment(ExperimentWrapper):
    def __init__(
        self,
        exp: Experiment,
        stages: list[Stage],
        ma_len,
        temp_error,
        preemptive_coef,
        ft_above,
        ft_below,
        ramping_rate,
    ):
        global running
        running = False
        self.exp = exp
        self.stages = stages

        self.ma_len = ma_len
        self.TEMP_ERROR = temp_error
        self.PREEMPTIVE_COEF = preemptive_coef
        self.FINETUNE_ABOVE = ft_above
        self.FINETUNE_BELOW = ft_below
        self.power_ramping_rate = ramping_rate

    def run(self, controller, verbose=True):
        global running

        with self.exp as exp:
            phase = 0
            stage = self.stages[phase]
            Tgoal = stage.temp
            prev_Tgoal = 0

            controller.root.after(
                0,
                lambda: controller.eq_lab.config(
                    text=f'Temperature Not Stable (goal = {Tgoal} °C)',
                    bg='tomato',
                ),
            )

            if isinstance(stage.power, DeltaPower):
                print(
                    'Cannot use Delta as the first heating power. '
                    'Use Constant'
                )
                return

            if isinstance(stage, TimedProfileStage):
                self.Pgoal = stage.preheat_power
            else:
                self.Pgoal = stage.power.value

            prev_Pgoal = None
            tmp_Pgoal = None
            is_ramping_power = False

            Tprev = None
            ma = []
            is_at_eq = False
            eq_time = None
            hold_start_elapsed = None
            hold_runtime_power = None
            hold_entry_power = None

            first_valid_time = None
            first_valid_temp = None
            filtered_resistance = None
            filtered_temperature = None
            filtered_heating_rate = None
            previous_filtered_temperature = None
            previous_filtered_time = None
            profile_current_command = None
            last_profile_control_time = None

            for i, I, V, R, T, elapsed, dt in exp.run(
                controller,
                verbose,
            ):
                if Tprev is None:
                    Tprev = T
                if prev_Pgoal is None:
                    prev_Pgoal = I * V

                ma.append(T)
                if len(ma) > self.ma_len:
                    ma.pop(0)

                if (
                    isinstance(stage, TimedProfileStage)
                    and T >= stage.max_stage_temp
                ):
                    print(
                        f'FIRST-STAGE SAFETY STOP: '
                        f'{T:.1f} °C >= '
                        f'{stage.max_stage_temp:.1f} °C'
                    )
                    running = False
                    return

                # Update the first-stage temperature filter every sample.
                if (
                    isinstance(stage, TimedProfileStage)
                    and T > stage.valid_temp
                ):
                    if filtered_temperature is None:
                        filtered_temperature = T
                    else:
                        alpha_T = stage.temp_filter_alpha
                        filtered_temperature += (
                            alpha_T
                            * (T - filtered_temperature)
                        )

                    # Estimate the sensor-reading slope and pass it through
                    # a slow first-order filter. This avoids the short-period
                    # power vibration produced by the earlier rate controller.
                    if previous_filtered_temperature is None:
                        previous_filtered_temperature = (
                            filtered_temperature
                        )
                        previous_filtered_time = elapsed
                    else:
                        rate_dt = elapsed - previous_filtered_time

                        if rate_dt > 0:
                            instantaneous_rate = (
                                filtered_temperature
                                - previous_filtered_temperature
                            ) / rate_dt

                            if filtered_heating_rate is None:
                                filtered_heating_rate = (
                                    instantaneous_rate
                                )
                            else:
                                rate_alpha = (
                                    1.0
                                    - np.exp(
                                        -rate_dt
                                        / stage.rate_filter_tau_s
                                    )
                                )
                                filtered_heating_rate += (
                                    rate_alpha
                                    * (
                                        instantaneous_rate
                                        - filtered_heating_rate
                                    )
                                )

                            previous_filtered_temperature = (
                                filtered_temperature
                            )
                            previous_filtered_time = elapsed

                # The first-stage hold starts explicitly. It does not wait
                # for the generic equilibrium detector.
                if (
                    isinstance(stage, TimedProfileStage)
                    and not is_at_eq
                    and elapsed >= stage.target_time
                    and filtered_temperature is not None
                    and filtered_temperature
                    >= stage.hold_entry_temp
                ):
                    is_at_eq = True
                    eq_time = time.perf_counter()
                    hold_start_elapsed = elapsed

                    # Bumpless transfer: keep the exact power command
                    # that brought this sample to 320. Do not make a power
                    # step on the first hold sample.
                    hold_entry_power = stage.clip_power(
                        self.Pgoal
                    )
                    hold_runtime_power = hold_entry_power
                    self.Pgoal = hold_entry_power

                    print(
                        f'320 °C HOLD START: '
                        f'Tfiltered={filtered_temperature:.2f} °C, '
                        f't={elapsed:.2f} s, '
                        f'holding for {stage.duration:.1f} s'
                    )
                    controller.root.after(
                        0,
                        lambda: controller.eq_lab.config(
                            bg='lightgreen',
                            text=(
                                f'Holding {Tgoal} °C: '
                                f'{stage.duration:.1f} s remaining'
                            ),
                        ),
                    )

                # Generic equilibrium detection remains unchanged for
                # stages 2 and 3.
                if (
                    not isinstance(stage, TimedProfileStage)
                    and abs(np.mean(ma) - Tgoal)
                    < self.TEMP_ERROR
                    and not is_at_eq
                ):
                    is_at_eq = True
                    eq_time = time.perf_counter()
                    controller.root.after(
                        0,
                        lambda: controller.eq_lab.config(
                            bg='lightgreen',
                            text=(
                                f'Temperature Stable '
                                f'(goal = {Tgoal} °C)'
                            ),
                        ),
                    )
                    print(
                        f'HIT EQUILIBRIUM: '
                        f'goal temperature = {Tgoal} °C '
                        f'at logged time {elapsed:.2f} s'
                    )

                if not is_at_eq:
                    if isinstance(stage, TimedProfileStage):
                        if T <= stage.valid_temp:
                            requested_power = stage.preheat_power

                            if elapsed > stage.desired_valid_time:
                                requested_power += (
                                    stage.late_boost_w_per_s
                                    * (
                                        elapsed
                                        - stage.desired_valid_time
                                    )
                                )

                            requested_power = stage.clip_power(
                                requested_power
                            )
                        else:
                            if first_valid_time is None:
                                first_valid_time = elapsed
                                first_valid_temp = T
                                print(
                                    f'PYROMETER ACTIVE: '
                                    f'T = {T:.1f} °C '
                                    f'at t = {elapsed:.2f} s '
                                    f'(desired ≈ '
                                    f'{stage.desired_valid_time:.2f} s)'
                                )

                            Tcontrol = filtered_temperature
                            Tref = stage.target_temperature(
                                elapsed,
                                first_valid_time,
                                first_valid_temp,
                            )

                            should_update = (
                                last_profile_control_time is None
                                or elapsed
                                - last_profile_control_time
                                >= stage.control_update_s
                            )

                            if should_update:
                                temp_error = Tref - Tcontrol
                                if (
                                    abs(temp_error)
                                    <= stage.temp_deadband
                                ):
                                    temp_error = 0.0

                                # Keep the endpoint predictor consistent
                                # with the variable-start straight line.
                                prediction_start_time = first_valid_time

                                desired_rate = (
                                    stage.temp - first_valid_temp
                                ) / max(
                                    stage.target_time
                                    - prediction_start_time,
                                    1e-6,
                                )

                                rate_for_prediction = (
                                    filtered_heating_rate
                                    if filtered_heating_rate
                                    is not None
                                    else desired_rate
                                )

                                remaining_time = max(
                                    stage.target_time - elapsed,
                                    0.0,
                                )
                                predicted_final_temp = (
                                    Tcontrol
                                    + rate_for_prediction
                                    * remaining_time
                                )

                                endpoint_error = np.clip(
                                    stage.temp
                                    - predicted_final_temp,
                                    -stage.endpoint_error_limit_c,
                                    stage.endpoint_error_limit_c,
                                )

                                # Ramp the prediction term in gradually
                                # after the pyrometer first becomes valid.
                                # This prevents one noisy early derivative
                                # from causing a large power step.
                                endpoint_weight = np.clip(
                                    (
                                        elapsed
                                        - first_valid_time
                                    )
                                    / stage.endpoint_ramp_in_s,
                                    0.0,
                                    1.0,
                                )

                                endpoint_correction = np.clip(
                                    stage.kp_endpoint
                                    * endpoint_error,
                                    -stage.endpoint_correction_limit_w,
                                    stage.endpoint_correction_limit_w,
                                ) * endpoint_weight

                                ramp_request = (
                                    stage.power.value
                                    + stage.kp_temp * temp_error
                                    + endpoint_correction
                                )

                                # Full-path predictive rate correction:
                                # From the first valid pyrometer reading until
                                # the 35 s endpoint, continually compare the
                                # filtered actual rate with the rate required
                                # to finish at 320. This correction works in
                                # both directions:
                                #   positive -> sample is behind, add power
                                #   negative -> sample is ahead, reduce power
                                path_rate_correction = 0.0

                                if remaining_time > 0.0:
                                    required_rate_to_deadline = (
                                        stage.temp - Tcontrol
                                    ) / max(
                                        remaining_time,
                                        0.5,
                                    )

                                    actual_rate_for_deadline = (
                                        rate_for_prediction
                                    )
                                    rate_error_to_deadline = (
                                        required_rate_to_deadline
                                        - actual_rate_for_deadline
                                    )

                                    # The derivative estimate is least reliable
                                    # immediately after the pyrometer activates.
                                    # Apply a short smooth ramp-in, then keep the
                                    # correction active for the entire path.
                                    prediction_reliability = np.clip(
                                        (
                                            elapsed
                                            - first_valid_time
                                        )
                                        / stage.endpoint_ramp_in_s,
                                        0.0,
                                        1.0,
                                    )

                                    path_rate_correction = np.clip(
                                        8.0
                                        * rate_error_to_deadline
                                        * prediction_reliability,
                                        -12.0,
                                        12.0,
                                    )

                                    ramp_request += (
                                        path_rate_correction
                                    )

                                # Keep following the linear trajectory
                                # through the final approach. Do not blend
                                # toward the fixed 88 W value before the
                                # target is reached; that caused the large
                                # jump/dip around 320.
                                requested_power = ramp_request

                                if Tcontrol >= stage.brake_temp:
                                    terminal_fraction = np.clip(
                                        (
                                            Tcontrol
                                            - stage.brake_temp
                                        )
                                        / (
                                            stage.temp
                                            - stage.brake_temp
                                        ),
                                        0.0,
                                        1.0,
                                    )
                                    positive_rate = max(
                                        filtered_heating_rate
                                        if filtered_heating_rate
                                        is not None
                                        else 0.0,
                                        0.0,
                                    )
                                    ahead_error = max(
                                        Tcontrol - Tref,
                                        0.0,
                                    )

                                    # Gentle predictive braking only.
                                    terminal_trim = terminal_fraction * (
                                        0.75 * positive_rate
                                        + 0.75 * ahead_error
                                    )
                                    requested_power -= min(
                                        terminal_trim,
                                        10.0,
                                    )

                                # Replace the former hard 65 W cap with a
                                # smooth bounded trim if the reading exceeds
                                # the target. The normal slew limiter remains
                                # active, so there is no abrupt power jump.
                                if Tcontrol >= stage.temp + 1.0:
                                    overshoot_trim = min(
                                        12.0,
                                        4.0
                                        * (
                                            Tcontrol
                                            - stage.temp
                                        ),
                                    )
                                    requested_power -= overshoot_trim

                                requested_power = stage.clip_power(
                                    requested_power
                                )
                                last_profile_control_time = elapsed
                            else:
                                requested_power = self.Pgoal

                            if verbose:
                                print(
                                    f'FIRST-STAGE TRACK: '
                                    f'Tref={Tref:.2f} °C, '
                                    f'Traw={T:.2f} °C, '
                                    f'Tfiltered='
                                    f'{Tcontrol:.2f}, '
                                    f'Tpred35='
                                    f'{predicted_final_temp:.2f}, '
                                    f'Prate='
                                    f'{path_rate_correction:.2f} W, '
                                    f'Pcmd='
                                    f'{requested_power:.2f} W'
                                )

                        self.Pgoal = stage.slew_limited_power(
                            self.Pgoal,
                            requested_power,
                            dt,
                        )

                    else:
                        # Original later-stage controller.
                        Trate = (T - Tprev) / dt
                        Tpred = T + Trate
                        pred_error = abs(Tpred - Tgoal)

                        if Tgoal > prev_Tgoal and Tpred > Tgoal:
                            self.Pgoal -= max(
                                0,
                                self.PREEMPTIVE_COEF
                                * pred_error,
                            )
                            is_ramping_power = False
                        elif Tgoal < prev_Tgoal and Tpred < Tgoal:
                            print(
                                'WARNING: controlled decreases in '
                                'temperature are not supported'
                            )
                            self.Pgoal += (
                                self.PREEMPTIVE_COEF
                                * pred_error
                            )

                else:
                    if isinstance(stage, TimedProfileStage):
                        Tcontrol = (
                            filtered_temperature
                            if filtered_temperature is not None
                            else T
                        )

                        hold_error = stage.temp - Tcontrol
                        if abs(hold_error) <= stage.hold_deadband:
                            hold_error = 0.0

                        if hold_entry_power is None:
                            hold_entry_power = stage.clip_power(
                                self.Pgoal
                            )
                        if hold_runtime_power is None:
                            hold_runtime_power = hold_entry_power

                        hold_rate = (
                            filtered_heating_rate
                            if filtered_heating_rate is not None
                            else 0.0
                        )

                        # Center the short hold on the actual entry power,
                        # not on the fixed 88 W nominal value.
                        requested_power = (
                            hold_entry_power
                            + 1.25 * hold_error
                            - 0.60 * max(
                                hold_rate,
                                0.0,
                            )
                        )

                        # Allow only a small reduction during the 3 s hold.
                        hold_floor = max(
                            stage.min_power,
                            hold_entry_power - 8.0,
                        )
                        hold_ceiling = min(
                            stage.max_power,
                            hold_entry_power + 5.0,
                        )
                        requested_power = float(
                            np.clip(
                                requested_power,
                                hold_floor,
                                hold_ceiling,
                            )
                        )

                        # Hold-specific slew: slow downward movement and
                        # faster recovery if the reading begins to dip.
                        if requested_power < self.Pgoal:
                            self.Pgoal = max(
                                requested_power,
                                self.Pgoal - 3.0 * dt,
                            )
                        else:
                            self.Pgoal = min(
                                requested_power,
                                self.Pgoal + 10.0 * dt,
                            )

                        hold_elapsed = (
                            elapsed - hold_start_elapsed
                        )
                        hold_remaining = max(
                            0.0,
                            stage.duration - hold_elapsed,
                        )

                        controller.root.after(
                            0,
                            lambda remaining=hold_remaining: (
                                controller.eq_lab.config(
                                    bg='lightgreen',
                                    text=(
                                        f'Holding {Tgoal} °C: '
                                        f'{remaining:.1f} s '
                                        f'remaining'
                                    ),
                                )
                            ),
                        )
                    else:
                        # Original later-stage hold correction.
                        error = abs(Tgoal - T)
                        if T < Tgoal:
                            self.Pgoal += (
                                self.FINETUNE_BELOW * error
                            )
                        elif T > Tgoal:
                            self.Pgoal -= (
                                self.FINETUNE_ABOVE * error
                            )

                if isinstance(stage, TimedProfileStage):
                    stage_complete = (
                        is_at_eq
                        and hold_start_elapsed is not None
                        and elapsed - hold_start_elapsed
                        >= stage.duration
                    )
                else:
                    stage_complete = (
                        eq_time is not None
                        and time.perf_counter() - eq_time
                        > self.stages[phase].duration
                    )

                if stage_complete:
                    if isinstance(stage, TimedProfileStage):
                        print(
                            f'320 °C HOLD COMPLETE: '
                            f'{elapsed - hold_start_elapsed:.2f} s'
                        )

                    ma = []
                    is_at_eq = False
                    eq_time = None
                    hold_start_elapsed = None
                    hold_runtime_power = None
                    hold_entry_power = None
                    phase += 1

                    if phase == len(self.stages):
                        running = False
                        print('Successful completion')
                        return

                    completed_stage = self.stages[phase - 1]
                    stage = self.stages[phase]
                    Tgoal = stage.temp
                    prev_Tgoal = completed_stage.temp
                    prev_Pgoal = self.Pgoal
                    tmp_Pgoal = None
                    is_ramping_power = False

                    # The successful first stage requires 0.10 s sampling.
                    # The user's original stage-2 and stage-3 controller ran
                    # with a true 0.25 s experiment loop. Restore that exact
                    # loop timing after the 320 °C hold, rather than trying
                    # to imitate it inside the 0.10 s loop.
                    if isinstance(completed_stage, TimedProfileStage):
                        exp.dt = 0.25
                        exp.next_time = time.perf_counter()
                        print(
                            'FIRST STAGE COMPLETE: restoring original '
                            '0.25 s loop for stages 2 and 3'
                        )

                    first_valid_time = None
                    first_valid_temp = None
                    filtered_resistance = None
                    filtered_temperature = None
                    filtered_heating_rate = None
                    previous_filtered_temperature = None
                    previous_filtered_time = None
                    profile_current_command = None
                    last_profile_control_time = None

                    if isinstance(stage.power, DeltaPower):
                        self.Pgoal += stage.power.value
                    elif isinstance(stage, TimedProfileStage):
                        self.Pgoal = stage.preheat_power
                    elif isinstance(stage.power, ConstantPower):
                        self.Pgoal = stage.power.value

                    controller.root.after(
                        0,
                        lambda: controller.eq_lab.config(
                            text=(
                                f'Temperature Not Stable '
                                f'(goal = {Tgoal} °C)'
                            ),
                            bg='tomato',
                        ),
                    )

                if not isinstance(stage, TimedProfileStage):
                    if (
                        tmp_Pgoal is not None
                        and is_ramping_power
                    ):
                        tmp_Pgoal += self.power_ramping_rate
                        self.Pgoal = tmp_Pgoal

                        if (
                            isinstance(
                                stage.power,
                                ConstantPower,
                            )
                            and self.Pgoal
                            >= stage.power.value
                        ):
                            self.Pgoal = stage.power.value
                            is_ramping_power = False

                    if (
                        self.Pgoal - prev_Pgoal > 150
                        and tmp_Pgoal is None
                        and not is_ramping_power
                    ):
                        is_ramping_power = True
                        tmp_Pgoal = max(
                            50,
                            prev_Pgoal
                            + self.power_ramping_rate,
                        )
                        self.Pgoal = tmp_Pgoal

                if self.Pgoal <= 0:
                    print(
                        'ERROR: Power dropped too low. Exiting...'
                    )
                    running = False
                    return

                if isinstance(stage, TimedProfileStage):
                    if filtered_resistance is None:
                        filtered_resistance = R
                    else:
                        alpha_R = (
                            stage.resistance_filter_alpha
                        )
                        filtered_resistance += (
                            alpha_R
                            * (R - filtered_resistance)
                        )

                    # Use only filtered resistance for the current
                    # calculation. The former instantaneous-power
                    # feedback amplified V/I measurement noise.
                    requested_current = np.sqrt(
                        self.Pgoal / filtered_resistance
                    )

                    if profile_current_command is None:
                        profile_current_command = exp.Ireq

                    profile_current_command = (
                        stage.slew_limited_current(
                            profile_current_command,
                            requested_current,
                            dt,
                        )
                    )
                    exp.setI(profile_current_command)
                else:
                    exp.setI(np.sqrt(self.Pgoal / R))

                Tprev = T

            controller.root.after(
                0,
                lambda: controller.eq_lab.config(
                    bg='lightgray',
                    text='Temperature Stability',
                ),
            )

    def to_dict(self):
        return {
            'NAME': 'SequentialExperiment',
            'stages': [
                {
                    'temperature': s.temp,
                    'duration': s.duration,
                    'power': str(s.power),
                    'stage_type': type(s).__name__,
                }
                for s in self.stages
            ],
            'ma_len': self.ma_len,
            'TEMP_ERROR': self.TEMP_ERROR,
            'PREEMPTIVE_COEF': self.PREEMPTIVE_COEF,
            'FINETUNE_ABOVE': self.FINETUNE_ABOVE,
            'FINETUNE_BELOW': self.FINETUNE_BELOW,
            **self.exp.to_dict(),
        }

    def data_to_csv(self, path):
        return pd.DataFrame.from_dict({
            'time_points': self.exp.time_points,
            'T_measured': self.exp.T_measured,
            'T_filtered': self.exp.T_filtered,
            'P_measured': self.exp.P_measured,
            'P_filtered': self.exp.P_filtered,
            'R_measured': self.exp.R_measured,
            'R_filtered': self.exp.R_filtered,
            'V_measured': self.exp.V_measured,
            'I_measured': self.exp.I_measured,
            'I_requested': self.exp.I_requested,
        }, orient='columns').to_csv(path)

    def params_to_csv(self, path):
        return pd.DataFrame.from_dict({
            'NAME': 'SequentialExperiment',
            'stage_types': [
                type(s).__name__ for s in self.stages
            ],
            'eq_temps': [
                s.temp for s in self.stages
            ],
            'eq_durations': [
                s.duration for s in self.stages
            ],
            'heating_powers': [
                str(s.power) for s in self.stages
            ],
            'ma_len': self.ma_len,
            'TEMP_ERROR': self.TEMP_ERROR,
            'PREEMPTIVE_COEF': self.PREEMPTIVE_COEF,
            'FINETUNE_ABOVE': self.FINETUNE_ABOVE,
            'FINETUNE_BELOW': self.FINETUNE_BELOW,
            'duration': self.exp.duration,
            'MAX_CURRENT': self.exp.MAX_CURRENT,
            'MAX_TEMP': self.exp.MAX_TEMP,
            'MAX_VOLTAGE': self.exp.MAX_VOLTAGE,
            'MIN_RESISTANCE': self.exp.MIN_RESISTANCE,
            'I0': self.exp.I0,
            'dt': self.exp.dt,
        }, orient='columns').to_csv(path)


## Initialization

In [10]:
system = System('COM15', 9600, 'COM12', 115200)
dcps, pyro = system.dcps, system.pyro

b'MATRIX,MPS-7500S-30-250,SN:116325100071,V2.200'


In [11]:
system.mc(), system.mv(), system.read_temp()

(0.0, 0.0, 200.0)

In [12]:
system.fast_pc(3)

#### C-C Composites Setup — full-path predictive linear control

Each sample begins its straight-line trajectory at its actual first valid pyrometer reading and targets the 320 calibration reading at 35 s. From that first valid reading onward, the controller continuously compares the filtered heating rate with the rate required to finish at 320. It adds bounded power when behind and subtracts bounded power when ahead. A short smooth ramp-in prevents an unreliable first derivative from causing a power step. The bumpless 3 s hold and stages 2–3 remain unchanged.


In [ ]:
# The first stage now uses a STRAIGHT-LINE pyrometer trajectory.
# It starts at the actual first valid pyrometer reading and targets the
# 320 calibration reading at the fixed 35 s deadline. The profile arrays
# remain only for compatibility; linear_target=True ignores their shape.
reference_profile_times = [
    8.8107,
    11.3107,
    13.8107,
    16.3107,
    18.8107,
    21.3107,
    23.8107,
    26.3107,
    28.8107,
    31.3107,
    33.8107,
    36.3107,
    38.0000,
    39.0592,
]

reference_profile_temps = [
    203.3,
    215.4,
    227.7,
    242.6,
    255.3,
    266.0,
    274.3,
    282.1,
    290.7,
    298.4,
    304.6,
    310.2,
    316.1,
    320.0,
]

FIRST_STAGE_DESIRED_VALID_TIME_S = 7.0
FIRST_STAGE_TARGET_TIME_S = 35.0

# Reduced from 163.6 W because the failed test entered the pyrometer range
# at 4.81 s. It ramps upward only if the sensor remains at 200 °C after 7 s.
FIRST_STAGE_PREHEAT_POWER_W = 140.0
FIRST_STAGE_BASE_POWER_W = 140.0
FIRST_STAGE_HOLD_POWER_W = 88.0

print(
    f'First-stage preheat power = '
    f'{FIRST_STAGE_PREHEAT_POWER_W:.1f} W'
)
print(
    f'First-stage pyrometer target = 320 at approximately '
    f'{FIRST_STAGE_TARGET_TIME_S:.1f} s'
)

exp = SequentialExperiment(
    Experiment(
        system,
        duration=1000,
        dt=0.10,
        I0=5,
        startup_power=65.4,
        startup_probe_current=5.0,
        startup_probe_duration=0.2,
        startup_duration=1.0,
        max_current=150,
        max_voltage=25,
        max_temp=1350,
        min_resistance=0.1,
    ),
    stages=[
        TimedProfileStage(
            ConstantPower(FIRST_STAGE_BASE_POWER_W),
            temp=320,
            duration=3,
            profile_times=reference_profile_times,
            profile_temps=reference_profile_temps,
            desired_valid_time=FIRST_STAGE_DESIRED_VALID_TIME_S,
            target_time=FIRST_STAGE_TARGET_TIME_S,
            preheat_power=FIRST_STAGE_PREHEAT_POWER_W,
            min_power=55.0,
            max_power=160.0,
            hold_power=FIRST_STAGE_HOLD_POWER_W,
            kp_temp=2.0,
            kp_hold=1.0,
            brake_temp=319.0,
            overshoot_power=65.0,
            max_stage_temp=330.0,
            slew_up_w_per_s=8.0,
            slew_down_w_per_s=30.0,
            late_boost_w_per_s=4.0,
            valid_temp=200.5,
            resistance_filter_alpha=0.08,
            current_slew_up_a_per_s=3.0,
            current_slew_down_a_per_s=15.0,
            temp_filter_alpha=0.15,
            temp_deadband=0.5,
            hold_deadband=0.35,
            hold_entry_temp=319.8,
            control_update_s=0.20,
            linear_target=True,
            kp_endpoint=0.30,
            endpoint_correction_limit_w=18.0,
            endpoint_error_limit_c=50.0,
            rate_filter_tau_s=2.5,
            endpoint_ramp_in_s=4.0,
        ),
        Stage(DeltaPower(20), 325, 40),
        Stage(ConstantPower(350), 400, 40),
    ],
    ma_len=8,
    preemptive_coef=1.3,
    temp_error=1,
    ft_above=1.0,
    ft_below=0.3,
    ramping_rate=20,
)

ExperimentController(exp).run()


First-stage preheat power = 140.0 W
First-stage pyrometer target = 320 at approximately 35.0 s
Initializing Experiment...
Turning on DCPS Output...  ✅
Initializing Current...   R0 = 0.1611 Ohm, Pstartup = 65.4 W, I = 20.15 V = 3.207
⚠️ Warning: -0.008240799885243177 seconds behind
(i=2) t = 0.29 | ∆t = 0.13 --> T = 200.00	I_req = 20.80	I = 20.56	V = 3.21	R = 0.16	P = 65.92
⚠️ Warning: -0.025621799752116203 seconds behind
(i=3) t = 0.42 | ∆t = 0.13 --> T = 200.00	I_req = 21.20	I = 20.56	V = 3.29	R = 0.16	P = 67.64
⚠️ Warning: -0.036417999770492315 seconds behind
(i=4) t = 0.55 | ∆t = 0.13 --> T = 200.00	I_req = 21.58	I = 21.16	V = 3.29	R = 0.16	P = 69.62
⚠️ Warning: -0.02444899990223348 seconds behind
(i=5) t = 0.68 | ∆t = 0.12 --> T = 200.00	I_req = 21.98	I = 21.39	V = 3.41	R = 0.16	P = 72.90
⚠️ Warning: -0.03089020005427301 seconds behind
(i=6) t = 0.81 | ∆t = 0.13 --> T = 200.00	I_req = 22.35	I = 22.19	V = 3.41	R = 0.15	P = 75.62
⚠️ Warning: -0.0344585997518152 seconds behind
(i=7) t

### Plots

In [ ]:
fig, axs = plt.subplots(nrows=5, sharex=True)
ax = axs[0]
ax.plot(exp.exp.time_points, exp.exp.T_measured)
# for Tgoal in exp.eq_temps:
    # ax.axhline(Tgoal, linestyle='dashed', color='black', linewidth=0.75)
# max = (exp.exp.time_points[np.argmax(exp.exp.T_measured)], np.max(exp.exp.T_measured))
# ax.annotate(fr'$T_{{max}} = {np.max(exp.exp.T_measured):.2f}$', (max[0] - 2, max[1] - 40))
ax.set_ylabel('Temperature (°C)')
# ax.set_yticks(exp.eq_temps)
ax = axs[1]
ax.plot(exp.exp.time_points, exp.exp.P_measured, color='green')
ax.set_ylabel('Power (Watts)')
ax = axs[2]
ax.plot(exp.exp.time_points, exp.exp.R_measured, color='orange')
ax.set_ylabel('Calculated Resistance')
ax.set_xlabel('Time (s)')
ax = axs[3]
ax.plot(exp.exp.time_points, exp.exp.V_measured, color='yellow')
ax.set_ylabel('Voltage')
ax.set_xlabel('Time (s)')
ax = axs[4]
ax.plot(exp.exp.time_points, exp.exp.I_measured, color='red')
ax.set_ylabel('Voltage')
ax.set_xlabel('Time (s)')
ax.set_xlim(left=40)
fig.tight_layout()
fig.set_figwidth(14)
fig.set_figheight(5)
fig.subplots_adjust(hspace=0.05)

In [ ]:
# multiphase.params_to_csv('07152025')
# multiphase.data_to_csv('07152025')